研究テーマ：独学者向けの機械翻訳システムの開発

構想：WikipediaなどのWebページを機械翻訳で独学者のレベルに合わせて学習用の文章を生成する

（例：英語からやさしい日本語に翻訳する、が、「やさしい」の定義が未定）

現時点の考え：Wikipediaからデータを取り、原文の難しさがいかに翻訳結果を影響するのかを検証

（英語版のデータ量が膨大すぎるため、こちらの課題ではやさしい英語版のものだけ扱います）

# Setup

Initialize output location

In [1]:
from pathlib import Path

OUT = Path("_out")
OUT.mkdir(exist_ok=True)

Check /content of session

In [2]:
!ls -lah /content

total 20K
drwxr-xr-x 1 root root 4.0K Oct 28 10:10 .
drwxr-xr-x 1 root root 4.0K Oct 28 10:01 ..
drwxr-xr-x 4 root root 4.0K Oct 26 22:18 .config
drwxr-xr-x 2 root root 4.0K Oct 28 10:10 _out
drwxr-xr-x 1 root root 4.0K Oct 26 22:18 sample_data


# 261014 データ収集
Webページの取得・データ整形を、授業で扱ったものをアレンジするか、自身の研究テーマに置き換えて実行した結果を共有してください。

In [ ]:
import requests
from bs4 import BeautifulSoup

# Target URL to scrape
url = "https://en.wikipedia.org/wiki/Wikipedia"

try:
    # Send a GET request to download the HTML
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    response.raise_for_status() # Check for request errors

    # Parse the HTML using BeautifulSoup
    soup = BeautifulSoup(response.text, "html.parser")

    # Print the page title
    print("=== Page Title ===")
    print(soup.title.string if soup.title else "No title found")
    print("\n" + "="*40 + "\n")

    # Print the first 1000 characters of the formatted HTML structure
    print("=== Pretty-Printed HTML (Truncated) ===")
    print(soup.prettify()[:1000] + "\n... [truncated] ...")

except Exception as e:
    print(f"An error occurred while fetching the URL: {e}")

=== Page Title ===
Wikipedia - Wikipedia


=== Pretty-Printed HTML (Truncated) ===
<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard skin-underline-clientpref-browser" dir="ltr" lang="en">
 <head>
  <meta charset="utf-8"/>
  <title>
   Wikipedia - Wikipedia
  </title>
  <script>
   (function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disab

In [ ]:
# Save the raw HTML content to a file in the output directory
html_file_path = OUT / "wiki.html"

try:
    # We write response.text (the raw HTML) to the file
    html_file_path.write_text(response.text, encoding="utf-8")
    print(f"Successfully saved HTML to: {html_file_path}")
except NameError:
    print("Make sure you have run the scraping cell first to define the 'response' variable.")

Successfully saved HTML to: _out/wiki.html


In [ ]:
import pandas as pd
from bs4 import BeautifulSoup

# Load the saved HTML file and parse it
with open(html_file_path, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f.read(), "html.parser")

data = []

# Find all heading tags in the page to map structure
h2_elements = soup.find_all("h2")
h2s = [h2.get_text() for h2 in h2_elements]
h2s

['Contents',
 'History',
 'Collaborative editing',
 ' Policies and content',
 'Governance',
 'Community',
 'Language editions',
 'Reception',
 'Operation',
 'Access to content',
 'Cultural influence',
 'Related projects',
 'See also',
 'Notes',
 'References',
 'Further reading',
 'External links']

In [ ]:
# Find all paragraphs
p_elements = soup.find_all("p")
ps = [p.get_text() for p in p_elements]
ps

['Wikipedia is a free content online encyclopedia website in 363 languages of the world in which 347 languages are currently active and 16 are closed.[3] It is written and maintained by a  community of volunteers, known as Wikipedians. ',
 'Users can freely use, share and change it, without having to pay. It is also one of the biggest wiki organizations. People can choose to give money to the Wikimedia Foundation to fund Wikipedia and its sister projects. It is an open content website. This means anyone can copy or edit it and make changes to it as long as they follow the rules for copying or editing. ',
 'Wikipedia is owned by an United States organization, the Wikimedia Foundation, which is in San Francisco.',
 "Wikipedia's name is a combination of two words, wiki and encyclopedia.[4]",
 'Wikipedia was started on January 9, 2001, by Jimmy Wales and Larry Sanger as part of an earlier online encyclopedia named Nupedia. On January 15, 2001, Wikipedia became a separate website of its own

In [ ]:
from bs4 import Tag

content = soup.select_one("#mw-content-text .mw-parser-output")
for junk in content.select("span.mw-editsection, .navbox, .navbox-styles, "
                           "style, script, link, meta, .mw-empty-elt"):
    junk.decompose()

def child_tags(section, found):
    # Collect the tag names of a section's content, in order, including subsections.
    for child in section.children:
        if not isinstance(child, Tag):             # skip whitespace
            continue
        if child.name == "section":                # nested h3/h4 subsection → look inside
            child_tags(child, found)
        elif "mw-heading" in (child.get("class") or []):
            h = child.find(["h3", "h4", "h5"])     # record the subheading itself
            if h:
                found.append(h.name)
        else:
            found.append(child.name)
    return found

In [ ]:
# making sense of what else are there in a section of a wiki page
# see what noise are there and whether the section has text for translation

rows = []
for h2 in content.select("div.mw-heading2 > h2"):
    section = h2.find_parent("section")
    tags = child_tags(section, [])
    rows.append({
        "h2": h2.get_text(strip=True),
        "nested_elements": ", ".join(dict.fromkeys(tags)),   # unique, in order of appearance
        "n_p": sum(1 for p in section.find_all("p") if p.get_text(strip=True)),
    })

df_map = pd.DataFrame(rows)
display(df_map)

,h2,nested_elements,n_p
0,History,"figure, div, h3, p",15
1,Collaborative editing,"figure, h3, p, div",11
2,Policies and content,"div, table, p, ul, h3",4
3,Governance,"div, p, h3, h4",7
4,Community,"div, figure, p, h3",6
5,Language editions,"div, p, ul, table, figure, h3",9
6,Reception,"div, p, h3, table, h4",32
7,Operation,"h3, div, figure, p",14
8,Access to content,"div, h3, p, h4, figure",14
9,Cultural influence,"h3, p, h4, div, figure",25


# 261021 構造化データの処理
HTML、XMLからの情報抽出を、授業で扱ったものをアレンジするか、自身の研究テーマに置き換えて実行した結果を共有してください。

## Get data

### Sources

In [ ]:
orig = 'enwiki-20261001-pages-articles-multistream.xml.bz2'
simp = 'simplewiki-20261001-pages-articles-multistream.xml.bz2'

In [ ]:
# Constructing the corresponding index filenames
orig_index = orig.replace('.xml.bz2', '-index.txt.bz2')
simp_index = simp.replace('.xml.bz2', '-index.txt.bz2')

print(f"Original Dump: {orig}")
print(f"Original Index: {orig_index}\n")
print(f"Simple Dump: {simp}")
print(f"Simple Index: {simp_index}")

Original Dump: enwiki-20261001-pages-articles-multistream.xml.bz2
Original Index: enwiki-20261001-pages-articles-multistream-index.txt.bz2

Simple Dump: simplewiki-20261001-pages-articles-multistream.xml.bz2
Simple Index: simplewiki-20261001-pages-articles-multistream-index.txt.bz2


### Download dumps

In [3]:
# simp xml
!wget -P {OUT} https://dumps.wikimedia.org/simplewiki/20261001/simplewiki-20261001-pages-articles-multistream.xml.bz2

--2026-10-28 10:10:31--  https://dumps.wikimedia.org/simplewiki/20261001/simplewiki-20261001-pages-articles-multistream.xml.bz2
Resolving dumps.wikimedia.org (dumps.wikimedia.org)... 208.80.154.242, 2620:0:861:ed1a::3:242
Connecting to dumps.wikimedia.org (dumps.wikimedia.org)|208.80.154.242|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 387919857 (370M) [application/octet-stream]
Saving to: ‘_out/simplewiki-20261001-pages-articles-multistream.xml.bz2’

simplewiki-20261001 100%[===================>] 369.95M  4.91MB/s    in 74s     

2026-10-28 10:11:46 (4.99 MB/s) - ‘_out/simplewiki-20261001-pages-articles-multistream.xml.bz2’ saved [387919857/387919857]



In [4]:
# simp index
!wget -P {OUT} https://dumps.wikimedia.org/simplewiki/20261001/simplewiki-20261001-pages-articles-multistream-index.txt.bz2

--2026-10-28 10:11:46--  https://dumps.wikimedia.org/simplewiki/20261001/simplewiki-20261001-pages-articles-multistream-index.txt.bz2
Resolving dumps.wikimedia.org (dumps.wikimedia.org)... 208.80.154.242, 2620:0:861:ed1a::3:242
Connecting to dumps.wikimedia.org (dumps.wikimedia.org)|208.80.154.242|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5251971 (5.0M) [application/octet-stream]
Saving to: ‘_out/simplewiki-20261001-pages-articles-multistream-index.txt.bz2’

simplewiki-20261001 100%[===================>]   5.01M  4.61MB/s    in 1.1s    

2026-10-28 10:11:48 (4.61 MB/s) - ‘_out/simplewiki-20261001-pages-articles-multistream-index.txt.bz2’ saved [5251971/5251971]



In [5]:
root = '/content/_out/'
index = root + 'simplewiki-20261001-pages-articles-multistream-index.txt.bz2'
pages = root + 'simplewiki-20261001-pages-articles-multistream.xml.bz2'

### Unzip data

In [6]:
# Option 1: Using shell commands (fast and easy in Colab)
# This decompresses the file and removes the original .bz2 file.
# !bzip2 -d /content/_out/simplewiki-20261001-pages-articles-multistream-index.txt.bz2

# If you want to keep the original .bz2 file, use the -k flag:
# !bzip2 -dk /content/_out/simplewiki-20261001-pages-articles-multistream-index.txt.bz2

# Option 2: Using Python's built-in bz2 module
import bz2

def unzipbz(infn, outfn):
  try:
    with bz2.open(infn, "rb") as f_in:
        with open(outfn, "wb") as f_out:
            f_out.write(f_in.read())
    print(f"Successfully decompressed to: {outfn}")
  except FileNotFoundError:
      print(f"File not found: {infn}. Please make sure the file is downloaded.")

In [7]:
intxt = root + 'index.txt'
unzipbz(index, intxt)

Successfully decompressed to: /content/_out/index.txt


In [8]:
pgxml = root + 'pages.xml'
unzipbz(pages, pgxml)

Successfully decompressed to: /content/_out/pages.xml


## Exploring the downloaded files


Index .txt file

In [ ]:
# Read and print the first 10 lines of the index file
with open(intxt, 'r', encoding='utf-8') as f:
    for i in range(10):
        line = f.readline()
        if not line:
            break
        print(line.strip())

552:1:April
552:2:August
552:6:Art
552:8:A
552:9:Air
552:11:Wikipedia:Administrators
552:12:Autonomous communities of Spain
552:13:Alan Turing
552:14:Alanis Morissette
552:17:Adobe Illustrator


Pages .xml file

In [ ]:
# Open and print the first x lines of the raw XML file
with open(pgxml, 'r', encoding='utf-8') as f:
    for i in range(30):
        line = f.readline()
        if not line:
            break
        print(line, end='')

<mediawiki xmlns="http://www.mediawiki.org/xml/export-0.11/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.mediawiki.org/xml/export-0.11/ http://www.mediawiki.org/xml/export-0.11.xsd" version="0.11" xml:lang="en">
  <siteinfo>
    <sitename>Wikipedia</sitename>
    <dbname>simplewiki</dbname>
    <base>https://simple.wikipedia.org/wiki/Main_Page</base>
    <generator>MediaWiki 1.47.0-wmf.21</generator>
    <case>first-letter</case>
    <namespaces>
      <namespace key="-2" case="first-letter">Media</namespace>
      <namespace key="-1" case="first-letter">Special</namespace>
      <namespace key="0" case="first-letter" />
      <namespace key="1" case="first-letter">Talk</namespace>
      <namespace key="2" case="first-letter">User</namespace>
      <namespace key="3" case="first-letter">User talk</namespace>
      <namespace key="4" case="first-letter">Wikipedia</namespace>
      <namespace key="5" case="first-letter">Wikipedia talk</namespace>


## XML

https://www.w3schools.com/xml/xml_whatis.asp$0

In [15]:
import xml.etree.ElementTree as ET

# To avoid loading the massive XML file into memory, we stream it using iterparse
# and extract just the first <page> element to analyze.
egpage = ""
namespaces = {}

for event, elem in ET.iterparse(pgxml, events=('start-ns', 'end')):
    if event == 'start-ns':
        prefix, uri = elem
        namespaces[prefix] = uri
    elif event == 'end':
        # Once we find a page tag, convert it to a string representation and stop
        if elem.tag.endswith('page'):
            egpage = ET.tostring(elem, encoding='utf-8').decode('utf-8')
            break

egpage

'<ns0:page xmlns:ns0="http://www.mediawiki.org/xml/export-0.11/">\n    <ns0:title>April</ns0:title>\n    <ns0:ns>0</ns0:ns>\n    <ns0:id>1</ns0:id>\n    <ns0:revision>\n      <ns0:id>10861257</ns0:id>\n      <ns0:parentid>10730460</ns0:parentid>\n      <ns0:timestamp>2026-05-18T12:42:53Z</ns0:timestamp>\n      <ns0:contributor>\n        <ns0:username>~2026-29643-61</ns0:username>\n        <ns0:id>1806169</ns0:id>\n      </ns0:contributor>\n      <ns0:comment>/* Trivia */</ns0:comment>\n      <ns0:origin>10861257</ns0:origin>\n      <ns0:model>wikitext</ns0:model>\n      <ns0:format>text/x-wiki</ns0:format>\n      <ns0:text bytes="22079" sha1="6ujzqa7xrm2rn4s2oofjpedjl08gtt2" xml:space="preserve">{{monththisyear|4}}\n\'\'\'April\'\'\' (Apr.) is the fourth [[month]] of the [[year]] in the [[Julian calendar|Julian]] and [[Gregorian calendar]]s, and comes between [[March]] and [[May]]. It is one of four months to have 30 [[day]]s.\n\nApril always begins on the same day of the week as [[Jul

In [21]:
# Save the XML string content in `egpage` to a file named 'example.xml' inside the output directory
example_file_path = OUT / "example.xml"

with open(example_file_path, "w", encoding="utf-8") as f:
    f.write(egpage)

print(f"Successfully saved egpage to: {example_file_path}")

Successfully saved egpage to: _out/example.xml


Namespaces: https://www.w3schools.com/xml/xml_namespaces.asp$0

In [19]:
### Define Namespace: define a shorthand prefix mapping to the MediaWiki XML schema URI.
# => creates dict {prefix : XML schema URI}
NS = {'tei': 'http://www.mediawiki.org/xml/export-0.11/'}
sample_root = ET.fromstring(egpage)

# Extract and print all unique tag types in this XML
unique_tags = sorted(list(set(elem.tag for elem in sample_root.iter())))
print("\n--- タグの種類一覧 (Raw tags) ---")
for t in unique_tags:
    print(t)

# Cleaned version without namespaces for better readability
print("\n--- タグの種類一覧 (Cleaned tags) ---")
for t in unique_tags:
    print(t.split("}")[-1])


--- タグの種類一覧 (Raw tags) ---
{http://www.mediawiki.org/xml/export-0.11/}comment
{http://www.mediawiki.org/xml/export-0.11/}contributor
{http://www.mediawiki.org/xml/export-0.11/}format
{http://www.mediawiki.org/xml/export-0.11/}id
{http://www.mediawiki.org/xml/export-0.11/}model
{http://www.mediawiki.org/xml/export-0.11/}ns
{http://www.mediawiki.org/xml/export-0.11/}origin
{http://www.mediawiki.org/xml/export-0.11/}page
{http://www.mediawiki.org/xml/export-0.11/}parentid
{http://www.mediawiki.org/xml/export-0.11/}revision
{http://www.mediawiki.org/xml/export-0.11/}sha1
{http://www.mediawiki.org/xml/export-0.11/}text
{http://www.mediawiki.org/xml/export-0.11/}timestamp
{http://www.mediawiki.org/xml/export-0.11/}title
{http://www.mediawiki.org/xml/export-0.11/}username

--- タグの種類一覧 (Cleaned tags) ---
comment
contributor
format
id
model
ns
origin
page
parentid
revision
sha1
text
timestamp
title
username


In [20]:
print("ルート要素:", sample_root.tag)
print("ルートの属性:", sample_root.attrib)
print("名前空間を指定しない検索:", sample_root.find(".//text"))
print("名前空間を指定した段落数:", len(sample_root.findall(".//tei:text", NS)))

ルート要素: {http://www.mediawiki.org/xml/export-0.11/}page
ルートの属性: {}
名前空間を指定しない検索: None
名前空間を指定した段落数: 1
